# Distribution of shape features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

## All profiles

In [ ]:
X = pd.read_csv("X.all_profiles.csv")
y = pd.read_csv("shape_features.all_profiles.csv")
data = pd.concat([X, y], axis=1)

In [ ]:
slurries = ["G50", "G45", "G40", "G40+IPA"]

Cas = data["capillary_number"].unique()
Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

In [ ]:
for TARGET in y.columns:

    fig, axes = plt.subplots(
        1, len(slurries), sharex=True, sharey="row", figsize=(10, 5)
    )

    for slurry, ax in zip(slurries, axes):
        ok = data["slurry"] == slurry
        subdata = data[ok]

        for ca in np.sort(subdata["capillary_number"].unique()):
            ca_data = subdata[subdata["capillary_number"] == ca]
            rgt_values = np.sort(ca_data["gap_to_thickness_ratio"].unique())
            box_width = (
                0.6 * np.min(np.diff(rgt_values)) if len(rgt_values) > 1 else 0.1
            )
            color = cmap(norm(ca))
            box = ax.boxplot(
                [
                    ca_data.loc[
                        ca_data["gap_to_thickness_ratio"] == rgt, TARGET
                    ].dropna()
                    for rgt in rgt_values
                ],
                positions=rgt_values,
                widths=box_width,
                patch_artist=True,
                manage_ticks=False,
                boxprops={"facecolor": color, "edgecolor": color, "alpha": 0.65},
                whiskerprops={"color": color},
                capprops={"color": color},
                medianprops={"color": color},
                flierprops={
                    "marker": "o",
                    "markerfacecolor": color,
                    "markeredgecolor": color,
                    "markersize": 2,
                },
            )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(
        sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
    )
    cbar.set_label("Ca")
    quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
    nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
    cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
    cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

    fig.supxlabel("Rgt")
    fig.supylabel(TARGET)
    fig.show()

## Mean profiles

In [ ]:
X = pd.read_csv("X.mean_profiles.csv")
y = pd.read_csv("shape_features.mean_profiles.csv")
data = pd.concat([X, y], axis=1)

In [ ]:
slurries = ["G50", "G45", "G40", "G40+IPA"]

Cas = data["capillary_number"].unique()
Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

In [ ]:
for TARGET in y.columns:

    fig, axes = plt.subplots(
        1, len(slurries), sharex=True, sharey="row", figsize=(10, 5)
    )

    for slurry, ax in zip(slurries, axes):
        ok = data["slurry"] == slurry
        subdata = data[ok]

        for ca in subdata["capillary_number"].unique():
            ok = subdata["capillary_number"] == ca

            ax.scatter(
                subdata[ok]["gap_to_thickness_ratio"],
                subdata[ok][TARGET],
                color=cmap(norm(ca)),
            )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(
        sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
    )
    cbar.set_label("Ca")
    quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
    nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
    cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
    cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

    fig.supxlabel("Rgt")
    fig.supylabel(TARGET)
    fig.show()